# 29장 양자 전환 — 자원 추정식·노출 재계산·스크립트 분류기

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB29-1: 쇼어 알고리듬의 ECDLP 자원 — 뢰텔러 외(2017) 일반식에 n=256 대입, 네 편의 논리 자원 비교 (공부용)
import math

def roetteler_bound(n):
    """Roetteler et al. (2017) 초록의 상한식 — 논리 큐비트와 Toffoli 게이트 수."""
    qubits = 9 * n + 2 * math.ceil(math.log2(n)) + 10
    toffoli = 448 * n**3 * math.log2(n) + 4090 * n**3
    return qubits, toffoli

for n in (192, 224, 256, 384, 521):
    q, t = roetteler_bound(n)
    print(f"n={n:>3}비트: 논리 큐비트 상한 {q:>5},  Toffoli 상한 {t:.3e}")

q256, t256 = roetteler_bound(256)
print()
print("n=256 — 식의 상한 Toffoli", f"{t256:.4e}", "| 논문이 시뮬레이션으로 보고한 값 약 1.26e11 (식은 상한이라 조금 크다)")
print("아가왈 외(2018)가 대입한 값 1.28e11 은 식의 상한을 그대로 쓴 것 →", f"{t256/1.28e11:.3f}", "배")

papers = [  # (연도, 논문, 논리 큐비트, Toffoli 또는 T 게이트 수)
    (2017, "Roetteler 외 (P-256)",            2330, 1.26e11),
    (2018, "Aggarwal 외 (보조 4 포함)",        2334, 1.28e11),
    (2022, "Webber 외 입력 회로 (T 게이트)",   2871, 5.76e9),
    (2026, "Babbush 외 — 저큐비트 회로",       1200, 9.0e7),
    (2026, "Babbush 외 — 저게이트 회로",       1450, 7.0e7),
]
print()
print(f"{'연도':<6}{'논문':<34}{'논리 큐비트':>10}{'게이트 수':>12}{'2017 대비 게이트':>16}")
for y, name, lq, gates in papers:
    ratio = "단위 다름(T)" if "T 게이트" in name else f"{1.26e11/gates:.0f}배"
    print(f"{y:<6}{name:<34}{lq:>10,}{gates:>12.2e}{ratio:>16}")
print()
print("물리 큐비트는 여기서 계산하지 않는다 — 오류율·코드 사이클·반응 시간·연결성·목표 시간의 함수다(본문 2절 심화).")

n=192비트: 논리 큐비트 상한  1754,  Toffoli 상한 5.300e+10
n=224비트: 논리 큐비트 상한  2042,  Toffoli 상한 8.528e+10
n=256비트: 논리 큐비트 상한  2330,  Toffoli 상한 1.287e+11
n=384비트: 논리 큐비트 상한  3484,  Toffoli 상한 4.494e+11
n=521비트: 논리 큐비트 상한  4719,  Toffoli 상한 1.150e+12

n=256 — 식의 상한 Toffoli 1.2875e+11 | 논문이 시뮬레이션으로 보고한 값 약 1.26e11 (식은 상한이라 조금 크다)
아가왈 외(2018)가 대입한 값 1.28e11 은 식의 상한을 그대로 쓴 것 → 1.006 배

연도    논문                                    논리 큐비트       게이트 수     2017 대비 게이트
2017  Roetteler 외 (P-256)                    2,330    1.26e+11              1배
2018  Aggarwal 외 (보조 4 포함)                   2,334    1.28e+11              1배
2022  Webber 외 입력 회로 (T 게이트)                 2,871    5.76e+09        단위 다름(T)
2026  Babbush 외 — 저큐비트 회로                    1,200    9.00e+07           1400배
2026  Babbush 외 — 저게이트 회로                    1,450    7.00e+07           1800배

물리 큐비트는 여기서 계산하지 않는다 — 오류율·코드 사이클·반응 시간·연결성·목표 시간의 함수다(본문 2절 심화).


In [ ]:
# NB29-2: 기준일 스냅샷(2차 집계)의 유형별 노출 수량 — 동봉 파일을 읽고 소계·비율을 다시 계산한다 (공부용)
import json, os
from decimal import Decimal

FALLBACK = {  # data/exposed_output_types_2026-10-25.json 과 같은 값 (파일이 없는 환경용)
    "height": 972132, "created": "2026-10-19T10:28:55Z", "issued": "20100415.625",
    "rows": [("p2pk_script_type", "1715076.99695260"), ("taproot_script_type", "217063.18329201"),
             ("multisig_script_type", "416.60645555"), ("pubkeyhash_reuse", "1185672.15392868"),
             ("witness_v0_keyhash_reuse", "2098570.39816002"), ("scripthash_reuse", "1199814.71613438"),
             ("witness_v0_scripthash_reuse", "705381.38634311"), ("witness_unknown_reuse", "0.00000000")]}

try:
    with open(os.path.join("data", "exposed_output_types_2026-10-25.json"), encoding="utf-8") as f:
        snap = json.load(f)
    prim = snap["primary"]
    height, created = prim["snapshot_block_height"], prim["snapshot_created_at_utc"][:19]
    issued = Decimal(snap["denominator"]["issued_supply_btc"])
    rows = [(r["category"], r["btc"]) for r in prim["by_category"]]
    src = "동봉 파일"
except FileNotFoundError:
    height, created, issued = FALLBACK["height"], FALLBACK["created"], Decimal(FALLBACK["issued"])
    rows, src = FALLBACK["rows"], "상수(파일 없음)"

print(f"출처: {src} | 스냅샷 블록 {height:,} ({created}) | 2차 집계(제3자 SQL, 재실행 안 함) | 분모 = 일정 발행량 {issued:,} BTC(상한)")
print(f"{'범주':<30}{'BTC':>18}{'발행량 대비':>12}")
script_type = Decimal(0); reuse = Decimal(0)
for cat, btc in rows:
    v = Decimal(btc)
    if cat.endswith("script_type"): script_type += v
    else: reuse += v
    print(f"{cat:<30}{v:>18,.3f}{v / issued * 100:>11.3f}%")
total = script_type + reuse
print("-" * 60)
print(f"{'소계: 스크립트에 키가 있는 출력':<30}{script_type:>18,.3f}{script_type / issued * 100:>11.3f}%")
print(f"{'소계: 재사용으로 키·스크립트 공개':<30}{reuse:>18,.3f}{reuse / issued * 100:>11.3f}%")
print(f"{'합계':<30}{total:>18,.3f}{total / issued * 100:>11.3f}%")
print()
print("읽는 법: 분모가 유통량의 상한이므로 비율은 하한 쪽이다. 재사용 범주는 키 없는 스크립트도 노출로 세어 과대 쪽이다.")
print("체인 밖 노출(xpub·서명 메시지)과 멤풀 체류 중 노출은 세지 않는다. 가격 환산은 하지 않는다.")

출처: 동봉 파일 | 스냅샷 블록 972,132 (2026-10-19T10:28:55) | 2차 집계(제3자 SQL, 재실행 안 함) | 분모 = 일정 발행량 20,100,415.625 BTC(상한)
범주                                           BTC      발행량 대비
p2pk_script_type                   1,715,076.997      8.533%
taproot_script_type                  217,063.183      1.080%
multisig_script_type                     416.606      0.002%
pubkeyhash_reuse                   1,185,672.154      5.899%
witness_v0_keyhash_reuse           2,098,570.398     10.440%
scripthash_reuse                   1,199,814.716      5.969%
witness_v0_scripthash_reuse          705,381.386      3.509%
witness_unknown_reuse                      0.000      0.000%
------------------------------------------------------------
소계: 스크립트에 키가 있는 출력                 1,932,556.787      9.615%
소계: 재사용으로 키·스크립트 공개                5,189,438.655     25.818%
합계                                 7,121,995.441     35.432%

읽는 법: 분모가 유통량의 상한이므로 비율은 하한 쪽이다. 재사용 범주는 키 없는 스크립트도 노출로 세어 과대 쪽이다.
체인 밖 노출(xpub·서명 메시지)과 멤풀 체류

In [ ]:
# NB29-3: 잠금 스크립트의 앞머리로 출력 유형과 「받는 순간 공개키 노출 여부」를 가린다 (공부용 — 가짜 바이트, 실사용 금지)
GENESIS_SPK = bytes.fromhex(  # 12장 제네시스 코인베이스 출력의 scriptPubKey (67바이트): PUSH65 <공개키> OP_CHECKSIG
    "4104678afdb0fe5548271967f1a67130b7105cd6a828e03909a67962e0ea1f61deb649f6bc3f4cef38c4f35504e51ec112de5c384df7ba0b8d578a4c702b6bf11d5fac")

def classify(spk: bytes):
    """표준 잠금 스크립트 유형 → (유형, 받는 순간 키 노출?, 설명). BIP-360 표의 '장기 노출' 열과 같은 판정."""
    n = len(spk)
    if n in (67, 35) and spk[0] in (0x41, 0x21) and spk[-1] == 0xAC:
        return "P2PK", True, "공개키(65 또는 33바이트)가 스크립트에 그대로 있다"
    if n == 25 and spk[:3] == b"\x76\xa9\x14" and spk[-2:] == b"\x88\xac":
        return "P2PKH", False, "공개키의 HASH160(20바이트)만 있다 — 지출 때 키 공개, 재사용하면 취약"
    if n == 23 and spk[:2] == b"\xa9\x14" and spk[-1] == 0x87:
        return "P2SH", False, "스크립트의 HASH160만 있다 — 지출 때 스크립트 공개"
    if n == 22 and spk[:2] == b"\x00\x14":
        return "P2WPKH", False, "세그윗 v0, 키 해시 20바이트"
    if n == 34 and spk[:2] == b"\x00\x20":
        return "P2WSH", False, "세그윗 v0, 스크립트 해시 32바이트"
    if n == 34 and spk[:2] == b"\x51\x20":
        return "P2TR", True, "세그윗 v1(탭루트), 조정된 x-only 공개키 32바이트가 그대로 있다"
    if n == 34 and spk[:2] == b"\x52\x20":
        return "P2MR(제안, BIP-360 Draft)", False, "세그윗 v2 제안 — 스크립트 트리의 머클 루트 32바이트만, 키 경로 없음"
    if n >= 3 and 0x51 <= spk[0] <= 0x60 and spk[-1] == 0xAE:
        return "P2MS(맨 다중서명)", True, "OP_m <키들> OP_n OP_CHECKMULTISIG — 공개키들이 그대로 있다"
    return "비표준/미분류", None, "이 분류기는 표준 유형만 안다"

samples = [
    ("제네시스 출력(실제 바이트)", GENESIS_SPK),
    ("P2PKH 모양(가짜)",  b"\x76\xa9\x14" + bytes(20) + b"\x88\xac"),
    ("P2SH 모양(가짜)",   b"\xa9\x14" + bytes(20) + b"\x87"),
    ("P2WPKH 모양(가짜)", b"\x00\x14" + bytes(20)),
    ("P2WSH 모양(가짜)",  b"\x00\x20" + bytes(32)),
    ("P2TR 모양(가짜)",   b"\x51\x20" + bytes(32)),
    ("P2MR 모양(가짜)",   b"\x52\x20" + bytes(32)),
    ("P2MS 1-of-2 모양(가짜)", b"\x51" + b"\x21" + bytes(33) + b"\x21" + bytes(33) + b"\x52\xae"),
]
print(f"{'표본':<26}{'바이트':>6}  {'유형':<28}{'받는 순간 키 노출':<16}설명")
for name, spk in samples:
    kind, exposed, why = classify(spk)
    flag = {True: "예", False: "아니오*", None: "?"}[exposed]
    print(f"{name:<26}{len(spk):>6}  {kind:<28}{flag:<16}{why}")
print()
print("* 아니오 = 받기만 하면 안전. 지출하면 키·스크립트가 공개되므로 같은 주소를 다시 쓰면 취약해진다(BIP-360 표의 No* 주석).")
print("제네시스 출력의 공개키 앞 바이트:", GENESIS_SPK[1:2].hex(), "(04 = 비압축 65바이트) — 2009년부터 체인에 그대로 적혀 있다.")

표본                           바이트  유형                          받는 순간 키 노출      설명
제네시스 출력(실제 바이트)               67  P2PK                        예               공개키(65 또는 33바이트)가 스크립트에 그대로 있다
P2PKH 모양(가짜)                  25  P2PKH                       아니오*            공개키의 HASH160(20바이트)만 있다 — 지출 때 키 공개, 재사용하면 취약
P2SH 모양(가짜)                   23  P2SH                        아니오*            스크립트의 HASH160만 있다 — 지출 때 스크립트 공개
P2WPKH 모양(가짜)                 22  P2WPKH                      아니오*            세그윗 v0, 키 해시 20바이트
P2WSH 모양(가짜)                  34  P2WSH                       아니오*            세그윗 v0, 스크립트 해시 32바이트
P2TR 모양(가짜)                   34  P2TR                        예               세그윗 v1(탭루트), 조정된 x-only 공개키 32바이트가 그대로 있다
P2MR 모양(가짜)                   34  P2MR(제안, BIP-360 Draft)     아니오*            세그윗 v2 제안 — 스크립트 트리의 머클 루트 32바이트만, 키 경로 없음
P2MS 1-of-2 모양(가짜)            71  P2MS(맨 다중서명)                예               OP_m <키들> OP_n OP_CHECKMULTISIG — 공개키들이 그대로 있다

* 아니오 = 받기만